# Chapter 9 &mdash; State Elimination: Bypass Edges and the $m\times n$ Rule

**Concept 2 of the Chapter 9 decomposition:** *State Elimination: Bypass Edges, Self-Loops, and the $m\times n$ Rule*

Delete a state $s$ and replace each in&ndash;out pair by one edge labelled (incoming)(self-loop)$^*$(outgoing).

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-State-Elimination/Concept-State-Elimination.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_NFA2RE     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The core rule. To delete state $s$, look at every **incoming** edge $p \to s$ and every
**outgoing** edge $s \to q$. For each such pair add a **bypass** edge

$$p \xrightarrow{\;R_{ps}\,(R_{ss})^*\,R_{sq}\;} q$$

where $R_{ss}$ is $s$'s **self-loop** (or $\varepsilon$ if there is none). If an edge
$p\to q$ already exists, **union** the new label onto it.

With $m$ incoming and $n$ outgoing edges this creates **$m\times n$** new edges &mdash;
the source of the blow-up in Concept 3.

The self-loop starred in the middle is the part beginners forget; it is what lets the
deleted state be visited any number of times.

## 2. Definitions

### The three label constructors Jove provides

In [ ]:
print("form_concat_RE(re1, re2) -- concatenation")
print("form_alt_RE([r1, r2, ...]) -- union")
print("form_kleene_RE(re)      -- star")
print("RE2Str(RE)              -- render a label as text")

### The bypass rule, written out

In [ ]:
def bypass_label(Rps, Rss, Rsq):
    """the label of the edge that replaces p -> s -> q"""
    mid = form_kleene_RE(Rss) if Rss is not None else None
    e = Rps
    if mid is not None: e = form_concat_RE(e, mid)
    return form_concat_RE(e, Rsq)

<!-- nav-strip -->

---

&larr;&nbsp;[Ch9&nbsp;1.&nbsp;The GNFA: Adding `Real_I` and `Real_F` to Stand On](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-The-GNFA/Concept-The-GNFA.ipynb) &nbsp;&middot;&nbsp; [**Chapter 9** index](https://github.com/ganeshutah/Jove/blob/master/Chapter9/README.md) &nbsp;&middot;&nbsp; [Ch9&nbsp;3.&nbsp;Exponential Blow-Up in NFA-to-RE Conversion](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-Exponential-Blow-Up-NFA2RE/Concept-Exponential-Blow-Up-NFA2RE.ipynb)&nbsp;&rarr;

---

## 3. Tests

A state with a self-loop: the star is essential.

In [ ]:
N = md2mc('''NFA
I : 0 -> S
S : 1 -> S        !! the self-loop
S : 0 -> F
''')
Gn = mk_gnfa(N)
_, _, restr = del_gnfa_states(Gn)
print("RE :", restr)
D = min_dfa(nfa2dfa(re2nfa(restr)))
for s in ['00', '010', '0110', '01110', '0', '01']:
    print("   %-8r accepted? %s" % (s, accepts_dfa(D, s)))
assert accepts_dfa(D, '01110') and not accepts_dfa(D, '01')
print("\n0 1* 0 -- the starred self-loop is right there in the middle.")

**$m\times n$:** two in-edges and three out-edges make six bypass edges.

In [ ]:
def count_bypass(m, n): return m * n
for m, n in [(1,1), (2,3), (3,3), (4,5)]:
    print("m=%d incoming, n=%d outgoing -> %d new edges" % (m, n, count_bypass(m, n)))
assert count_bypass(2, 3) == 6

Existing edges are **unioned**, not overwritten.

In [ ]:
Two = md2mc('''NFA
I : 0 -> S
I : 1 -> F        !! a direct edge that already exists
S : 0 -> F
''')
_, _, r2 = del_gnfa_states(mk_gnfa(Two))
print("RE :", r2)
D = min_dfa(nfa2dfa(re2nfa(r2)))
assert accepts_dfa(D, '1') and accepts_dfa(D, '00')
print("both '1' (direct) and '00' (via S) are accepted -- the labels were unioned")

Deleting in a different **order** gives a different-looking but equivalent RE.

In [ ]:
N3 = md2mc('''NFA
I : 0 -> A
A : 1 -> B
B : 0 -> F
A : 0 -> A
''')
# DelList must be a permutation of ALL the original states; Real_I and
# Real_F are appended by del_gnfa_states itself.
G1 = mk_gnfa(N3); _, _, ra = del_gnfa_states(G1, DelList=['I', 'A', 'B', 'F'])
G2 = mk_gnfa(N3); _, _, rb = del_gnfa_states(G2, DelList=['F', 'B', 'A', 'I'])
print("delete I,A,B,F :", ra)
print("delete F,B,A,I :", rb)
Da, Db = min_dfa(nfa2dfa(re2nfa(ra))), min_dfa(nfa2dfa(re2nfa(rb)))
assert iso_dfa(Da, Db)
print("\ndifferent strings, same language :", iso_dfa(Da, Db))

## 4. Exercises


1. Delete a state with **no** self-loop by hand. What goes in the middle?
2. Why must the self-loop be starred rather than just concatenated?
3. Delete a state with 3 in-edges and 4 out-edges. How many labels do you write?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter9/Concept-State-Elimination')